# Lab 5: SageMaker Pipelines로 Bank Marketing ML 워크플로우 자동화

노트북 1~4의 **전처리 → 훈련 → 평가** 작업을 SageMaker Pipelines로 자동화합니다.

**파이프라인 구성:**

```
BankPreprocess
    ├─ BankTrainConservative  (병렬)
    └─ BankTrainAggressive    (병렬)
              └─ BankEvaluate
                      └─ BankAUCCondition
                              ├─ (AUC ≥ 임계값) BankRegisterModel
                              └─ (AUC <  임계값) BankPipelineFail
```

> ⚠️ **사전 조건**: `0-setup.ipynb`와 `1-preprocessing.ipynb`를 먼저 실행하여 환경 변수를 저장해야 합니다.

In [1]:
# ============================================================
# 패키지 설치 확인 및 커널 재시작 (무한 루프 방지 가드)
# ============================================================
# find_spec()으로 설치 여부를 먼저 확인합니다.
# 이미 설치된 경우 이 블록 전체를 건너뜁니다.
# 설치되지 않은 경우에만 pip install 후 커널을 재시작합니다.
#
# 주의: "Run All Cells" 실행 시 이 가드 없이 매번 설치+재시작이
#       반복되는 무한 루프가 발생할 수 있습니다.
#
# 핀 버전 이유:
#   setuptools<81  : setuptools 81+에서 pkg_resources 제거 → import mlflow(2.13.2) 실패
#   mlflow==2.13.2 : sagemaker-mlflow와 호환되는 버전
import importlib.util, sys

if importlib.util.find_spec("sagemaker.sklearn") is None:
    import subprocess
    # 필수 패키지를 조용히(-q) 설치합니다
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
        "setuptools<81", "sagemaker>=2.220,<3", "mlflow==2.13.2", "sagemaker-mlflow"])
    # 설치 후 커널을 재시작해야 새 패키지가 인식됩니다
    import IPython
    IPython.Application.instance().kernel.do_shutdown(True)

sagemaker.config INFO - Fetched defaults config from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix


In [2]:
import boto3           # AWS SDK: S3, SageMaker 클라이언트 생성
import sagemaker       # SageMaker Python SDK
import mlflow          # 실험 추적 및 모델 로깅
import os

# 파이프라인 세션: 파이프라인 컨텍스트에서 잡 정의 시 실제 실행 없이 DAG만 구성
from sagemaker.workflow.pipeline_context import PipelineSession
# 파이프라인 객체 및 DAG 관리
from sagemaker.workflow.pipeline import Pipeline
# 파이프라인 런타임 파라미터 타입 정의
from sagemaker.workflow.parameters import ParameterString, ParameterFloat
# 파이프라인 단계 타입: 전처리, 훈련, 모델 등록, 실패, 조건
from sagemaker.workflow.steps import ProcessingStep, TrainingStep
from sagemaker.workflow.model_step import ModelStep
from sagemaker.workflow.fail_step import FailStep
from sagemaker.workflow.condition_step import ConditionStep
# 조건 연산자: >= 비교 (AUC 임계값 체크)
from sagemaker.workflow.conditions import ConditionGreaterThanOrEqualTo, ConditionEquals
# 파이프라인 런타임 함수: 문자열 결합, JSON 경로 추출
from sagemaker.workflow.functions import Join, JsonGet
# 단계 출력 결과를 파이프라인 내에서 참조하기 위한 PropertyFile
from sagemaker.workflow.properties import PropertyFile
# 전처리 잡 입출력 및 스크립트 실행기
from sagemaker.processing import ProcessingInput, ProcessingOutput, ScriptProcessor
from sagemaker.sklearn import SKLearn
# FrameworkProcessor: SKLearn 등 프레임워크를 전처리에 활용하고 의존성 자동 설치
from sagemaker.processing import FrameworkProcessor
# 훈련 잡 S3 입력 설정
from sagemaker.inputs import TrainingInput
# 내장 알고리즘 Estimator (XGBoost)
from sagemaker.estimator import Estimator
# 내장 알고리즘 컨테이너 URI 조회
from sagemaker import image_uris
# 모델 객체 및 등록
from sagemaker.model import Model

print(f"SageMaker SDK: {sagemaker.__version__}")

SageMaker SDK: 2.257.3


## 1. 환경 변수 복원

`0-setup.ipynb`에서 `%store`로 저장된 환경 변수를 복원하고 세 종류의 세션을 초기화합니다.

| 변수 | 설명 |
|------|------|
| `bucket` | SageMaker 기본 S3 버킷 이름 |
| `prefix` | 실험 데이터/모델 저장 경로 접두사 |
| `role` | SageMaker 실행에 사용할 IAM 역할 ARN |
| `region` | AWS 리전 (예: us-west-2) |
| `mlflow_arn` | MLflow 추적 서버 ARN |
| `experiment_name` | MLflow 실험 이름 |
| `input_source` | 원본 Bank Marketing CSV 파일의 S3 URI |

> **세션 구분**: `sagemaker.Session`은 일반 작업에, `PipelineSession`은 파이프라인 단계 정의 전용으로 사용합니다. `PipelineSession`을 전달하면 Estimator·Processor의 `.fit()`/`.run()` 호출 시 실제 잡이 시작되지 않고 파이프라인 DAG 메타데이터만 수집됩니다.

In [3]:
# ============================================================
# 환경 변수 복원: 0-setup.ipynb에서 %store로 저장된 값을 가져옵니다
# ============================================================
%store -r bucket          # SageMaker 기본 S3 버킷
%store -r prefix          # 실험 데이터/모델 저장 경로
%store -r role            # SageMaker IAM 역할 ARN
%store -r region          # AWS 리전 코드
%store -r mlflow_arn      # MLflow 추적 서버 ARN (DataZone API로 조회된 값)
%store -r experiment_name # MLflow 실험 이름
%store -r input_source    # Bank Marketing CSV S3 URI

# 필수 변수가 모두 복원되었는지 확인합니다
# 누락된 경우 즉시 오류를 발생시켜 이후 셀 실행을 방지합니다
required = {"bucket": bucket, "prefix": prefix, "role": role,
            "region": region, "mlflow_arn": mlflow_arn,
            "experiment_name": experiment_name, "input_source": input_source}
missing = [k for k, v in required.items() if not v]
if missing:
    raise RuntimeError(f"누락된 변수: {missing}\n0-setup.ipynb와 1-preprocessing.ipynb를 먼저 실행하세요.")

print(f"bucket          : {bucket}")
print(f"prefix          : {prefix}")
print(f"region          : {region}")
print(f"mlflow_arn      : {mlflow_arn}")
print(f"experiment_name : {experiment_name}")
print(f"input_source    : {input_source}")

no stored variable or alias #
no stored variable or alias SageMaker
no stored variable or alias 기본
no stored variable or alias S3
no stored variable or alias 버킷
no stored variable or alias #
no stored variable or alias 실험
no stored variable or alias 데이터/모델
no stored variable or alias 저장
no stored variable or alias 경로
no stored variable or alias #
no stored variable or alias SageMaker
no stored variable or alias IAM
no stored variable or alias 역할
no stored variable or alias ARN
no stored variable or alias #
no stored variable or alias AWS
no stored variable or alias 리전
no stored variable or alias 코드
no stored variable or alias #
no stored variable or alias MLflow
no stored variable or alias 추적
no stored variable or alias 서버
no stored variable or alias ARN
no stored variable or alias (DataZone
no stored variable or alias API로
no stored variable or alias 조회된
no stored variable or alias 값)
no stored variable or alias #
no stored variable or alias MLflow
no stored variable or alias 실험
no st

In [4]:
# ============================================================
# AWS/SageMaker 세션 초기화
# ============================================================
# boto_session: 지정 리전의 AWS 세션 (S3, SageMaker API 호출 기반)
boto_session      = boto3.Session(region_name=region)

# sagemaker_session: 일반 SageMaker 작업용 세션
#   - 데이터 업로드, 모델 배포 등 파이프라인 외 작업에 사용
sagemaker_session = sagemaker.Session(boto_session=boto_session)

# pipeline_session: 파이프라인 단계 정의 전용 세션
#   - 이 세션으로 Estimator.fit() / Processor.run()을 호출하면
#     실제 잡이 실행되지 않고 파이프라인 DAG 구성 메타데이터만 수집합니다
pipeline_session  = PipelineSession(boto_session=boto_session)

# sm_client: 저수준 SageMaker API (파이프라인 상태 조회, 잡 상세 정보 등)
sm_client         = boto_session.client("sagemaker")

print("세션 초기화 완료")

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
세션 초기화 완료


## 2. 파이프라인 파라미터

파이프라인 실행 시 외부에서 주입 가능한 파라미터를 정의합니다. 기본값을 가지므로 `pipeline.start()` 호출 시 생략 가능하며, 코드 수정 없이 값만 바꿔 재실험할 수 있습니다.

| 파라미터 | 타입 | 기본값 | 용도 |
|----------|------|--------|------|
| `ProcessingInstanceType` | String | ml.m5.large | 전처리·평가 인스턴스 |
| `TrainingInstanceType` | String | ml.m5.large | 훈련 인스턴스 |
| `ModelApprovalStatus` | String | PendingManualApproval | 등록 모델 초기 승인 상태 |
| `AucThreshold` | Float | 0.75 | 모델 등록 최소 AUC 기준 |
| `InputData` | String | S3 URI | 원본 데이터 경로 |
| `MlflowTrackingArn` | String | mlflow_arn | MLflow 추적 서버 |
| `MlflowExperimentName` | String | experiment_name | 실험 이름 |

In [5]:
# ============================================================
# 파이프라인 파라미터 정의
# ============================================================
# pipeline.start()에서 값을 전달하지 않으면 default_value를 사용합니다.
# 코드 수정 없이 인스턴스 타입, 임계값 등을 외부에서 제어할 수 있습니다.

# 전처리 및 평가 잡에 사용할 EC2 인스턴스 타입
processing_instance_type = ParameterString(
    name="ProcessingInstanceType", default_value="ml.m5.large")

# 훈련 잡에 사용할 EC2 인스턴스 타입
training_instance_type = ParameterString(
    name="TrainingInstanceType", default_value="ml.m5.large")

# 모델 등록 후 승인 상태:
#   PendingManualApproval → 인간 검토 필요 (프로덕션 배포 전 안전장치)
#   Approved              → 즉시 배포 가능
model_approval_status = ParameterString(
    name="ModelApprovalStatus", default_value="PendingManualApproval")

# 모델 등록 기준 AUC 임계값: 이 값 미만이면 BankPipelineFail 단계로 분기
auc_threshold = ParameterFloat(
    name="AucThreshold", default_value=0.75)

# 원본 Bank Marketing CSV 데이터의 S3 URI
input_data = ParameterString(
    name="InputData", default_value=input_source)

# MLflow 추적 서버 ARN (DataZone 환경에서 조회된 값)
mlflow_tracking_arn = ParameterString(
    name="MlflowTrackingArn", default_value=mlflow_arn)

# MLflow 실험 이름 (노트북 0~4와 동일한 실험에 함께 로깅)
mlflow_experiment = ParameterString(
    name="MlflowExperimentName", default_value=experiment_name)

print("파이프라인 파라미터 정의 완료")

파이프라인 파라미터 정의 완료


## 3. 스크립트 작성

파이프라인의 각 단계에서 실행될 Python 스크립트를 `pipeline_code/` 디렉터리에 작성합니다.

- **`preprocessing.py`**: Bank Marketing CSV → 전처리 → train / validation / test / baseline CSV 분할 저장
- **`evaluation.py`**: 두 XGBoost 모델을 테스트셋으로 비교하고 `evaluation.json`에 메트릭 저장 + MLflow 로깅

> `%%writefile` 셀로 파일을 생성합니다. 노트북 1의 전처리 로직과 동일하지만, SageMaker Processing 컨테이너의 표준 경로(`/opt/ml/processing`)를 사용하도록 조정되었습니다.

In [6]:
# 파이프라인 스크립트 저장 디렉터리 생성 (이미 존재해도 오류 없음)
os.makedirs("pipeline_code", exist_ok=True)

In [7]:
%%writefile pipeline_code/preprocessing.py
"""
Bank Marketing 데이터 전처리 스크립트

노트북 1(1-preprocessing.ipynb)과 동일한 로직을 SageMaker Processing 잡으로 실행합니다.
SageMaker Processing 컨테이너의 표준 경로(/opt/ml/processing)를 사용합니다.

입력:  /opt/ml/processing/input/bank-additional-full.csv
출력:  /opt/ml/processing/output/
         train/train.csv           -- 훈련 데이터 (70%, 레이블 첫 컬럼, 헤더 없음)
         validation/validation.csv -- 검증 데이터 (20%, 레이블 첫 컬럼, 헤더 없음)
         test/test_x.csv           -- 테스트 피처 (10%, 헤더 없음)
         test/test_y.csv           -- 테스트 레이블 (10%)
         baseline/baseline.csv    -- Model Monitor 베이스라인용 (훈련과 동일)
"""
import os
import pandas as pd
from sklearn.model_selection import train_test_split

if __name__ == "__main__":
    # SageMaker Processing 컨테이너의 기본 데이터 경로
    base_dir = "/opt/ml/processing"

    # 쉼표(,) 구분자 CSV 로드 (이 S3 파일은 쉼표 구분자로 저장되어 있습니다)
    df = pd.read_csv(f"{base_dir}/input/bank-additional-full.csv", sep=",")

    # 컬럼명 정리: XGBoost는 '.' 포함 피처명을 처리하지 못하므로 '_'로 변환
    df.columns = [c.replace(".", "_") for c in df.columns]

    # 파생 변수 생성
    # pdays==999: 이전 캠페인 연락이 없었음을 의미하는 특수값 → 이진 플래그로 변환
    df["no_previous_contact"] = (df["pdays"] == 999).astype(int)
    # 비경제활동 인구 여부: 학생/은퇴자/실업자 → 이진 플래그
    df["not_working"] = df["job"].isin(["student", "retired", "unemployed"]).astype(int)

    # 불필요 컬럼 제거
    # duration: 실제 예측 시점에 알 수 없는 정보 누수(leakage) 컬럼
    # 경제 지표 컬럼: 노이즈 제거 목적
    drop_cols = ["duration", "emp_var_rate", "cons_price_idx",
                 "cons_conf_idx", "euribor3m", "nr_employed"]
    df.drop(columns=drop_cols, inplace=True)

    # 범주형 변수 원-핫 인코딩 후 정수형 변환 (XGBoost 입력 요건)
    df = pd.get_dummies(df).astype(int)

    # 레이블 컬럼(y_yes)을 첫 번째 컬럼으로 이동
    # XGBoost 내장 알고리즘은 첫 번째 컬럼을 레이블로 인식합니다
    label_col = "y_yes"
    cols = [label_col] + [c for c in df.columns if c != label_col]
    df = df[cols]

    # 클래스 불균형을 고려한 stratify 분할: 70% 훈련 / 20% 검증 / 10% 테스트
    # pd.get_dummies(df)는 y 컬럼(yes/no)에서 y_yes와 y_no 두 컬럼을 생성합니다.
    # label_col(y_yes)만 제거하면 y_no(= 1 - y_yes)가 피처로 남아 레이블 누수 발생.
    # y_ 로 시작하는 모든 컬럼을 피처에서 제거하여 누수를 방지합니다.
    X = df.drop(columns=[c for c in df.columns if c.startswith("y_")])
    y = df[label_col]

    # 1단계: 70/30 분할
    X_train, X_tmp, y_train, y_tmp = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y)
    # 2단계: 나머지 30%를 2:1로 분할 → 검증 20% / 테스트 10%
    X_val, X_test, y_val, y_test = train_test_split(
        X_tmp, y_tmp, test_size=1/3, random_state=42, stratify=y_tmp)

    # 훈련/검증 데이터: 레이블 첫 컬럼, 헤더 없는 CSV (XGBoost 내장 알고리즘 요건)
    train_df = pd.concat([y_train, X_train], axis=1)
    val_df   = pd.concat([y_val,   X_val],   axis=1)

    # 출력 디렉터리 생성
    os.makedirs(f"{base_dir}/output/train",      exist_ok=True)
    os.makedirs(f"{base_dir}/output/validation", exist_ok=True)
    os.makedirs(f"{base_dir}/output/test",       exist_ok=True)
    os.makedirs(f"{base_dir}/output/baseline",   exist_ok=True)

    # CSV 저장: header=False, index=False (XGBoost 내장 알고리즘 입력 형식)
    train_df.to_csv(f"{base_dir}/output/train/train.csv",           index=False, header=False)
    val_df.to_csv(  f"{base_dir}/output/validation/validation.csv", index=False, header=False)
    # 테스트 데이터: 피처(X)와 레이블(y)을 분리 저장 → 평가 스크립트에서 독립 로드
    X_test.to_csv(  f"{base_dir}/output/test/test_x.csv",           index=False, header=False)
    y_test.to_csv(  f"{base_dir}/output/test/test_y.csv",           index=False, header=False)
    # 베이스라인: Model Monitor 드리프트 감지의 기준선으로 사용
    train_df.to_csv(f"{base_dir}/output/baseline/baseline.csv",     index=False, header=False)

    print(f"train      : {len(train_df):,}행")
    print(f"validation : {len(val_df):,}행")
    print(f"test       : {len(X_test):,}행")

Overwriting pipeline_code/preprocessing.py


In [8]:
%%writefile pipeline_code/requirements.txt
pandas
numpy
scikit-learn

Overwriting pipeline_code/requirements.txt


In [9]:
%%writefile pipeline_code/evaluation.py
"""
두 XGBoost 모델 비교 평가 스크립트

BankTrainConservative(model1)와 BankTrainAggressive(model2)를 테스트셋으로 평가하고
더 나은 AUC를 기록한 모델을 선정합니다. 결과는 evaluation.json에 저장되며,
BankAUCCondition 단계가 이 파일을 읽어 모델 등록 여부를 결정합니다.

입력:  /opt/ml/processing/model1/model.tar.gz  -- Conservative 모델 아티팩트
       /opt/ml/processing/model2/model.tar.gz  -- Aggressive 모델 아티팩트
       /opt/ml/processing/test/test_x.csv      -- 테스트 피처
       /opt/ml/processing/test/test_y.csv      -- 테스트 레이블

출력:  /opt/ml/processing/evaluation/evaluation.json
"""
import os, json, tarfile
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import roc_auc_score, accuracy_score

# SageMaker Processing 컨테이너의 기본 데이터 경로
BASE = "/opt/ml/processing"


def load_booster(model_dir):
    """
    model.tar.gz 아카이브에서 XGBoost 부스터를 로드합니다.

    SageMaker 훈련 잡은 모델 아티팩트를 model.tar.gz로 압축해 S3에 저장합니다.
    파일명은 훈련 스크립트 규약에 따라 'xgboost-model'이어야 합니다.
    """
    tar_path = os.path.join(model_dir, "model.tar.gz")
    if os.path.exists(tar_path):
        # tar.gz 아카이브 압축 해제 (같은 디렉터리에)
        with tarfile.open(tar_path) as t:
            t.extractall(model_dir)
    booster = xgb.Booster()
    # XGBoost 내장 알고리즘이 저장하는 표준 모델 파일명
    booster.load_model(os.path.join(model_dir, "xgboost-model"))
    return booster


def main():
    # MLflow 의존성 설치 (XGBoost 컨테이너에 미포함)
    import subprocess, sys as _sys
    # XGBoost 컨테이너의 /miniconda3/ 구버전 mlflow와 충돌을 피하기 위해
    # --target으로 /tmp/ml_deps에 별도 설치 후 sys.path 앞에 추가합니다.
    # (--force-reinstall은 /miniconda3/을 덮지 못하고 sagemaker-mlflow가
    #  mlflow 2.13.2에 없는 MlflowTracingException을 요구하므로 최신 mlflow 사용)
    _ml_dir = "/tmp/ml_deps"
    subprocess.check_call([_sys.executable, "-m", "pip", "install", "-q",
                           "--target", _ml_dir, "mlflow", "sagemaker-mlflow"])
    if _ml_dir not in _sys.path:
        _sys.path.insert(0, _ml_dir)
    import importlib as _il; _il.invalidate_caches()

    # 헤더 없는 CSV 로드 (전처리 스크립트의 header=False 저장 방식과 일치)
    test_x = pd.read_csv(f"{BASE}/test/test_x.csv", header=None)
    test_y = pd.read_csv(f"{BASE}/test/test_y.csv", header=None).squeeze()
    # XGBoost 예측을 위한 DMatrix 변환 (메모리 효율적 내부 포맷)
    dtest  = xgb.DMatrix(test_x)

    # 두 모델 로드
    model1 = load_booster(f"{BASE}/model1")  # Conservative (eta=0.5, num_round=50)
    model2 = load_booster(f"{BASE}/model2")  # Aggressive  (eta=0.1, num_round=100)

    # 예측 확률값 계산 (binary:logistic → 0~1 사이 양성 클래스 확률)
    pred1 = model1.predict(dtest)
    pred2 = model2.predict(dtest)

    # AUC: ROC 곡선 아래 넓이 (클수록 분류 성능이 좋음, 1.0이 완벽)
    auc1 = float(roc_auc_score(test_y, pred1))
    auc2 = float(roc_auc_score(test_y, pred2))
    # Accuracy: 0.5 임계값으로 이진 분류 후 정확도 계산
    acc1 = float(accuracy_score(test_y, (pred1 > 0.5).astype(int)))
    acc2 = float(accuracy_score(test_y, (pred2 > 0.5).astype(int)))

    # AUC 기준으로 더 나은 모델 선택
    best_model_num = 1 if auc1 >= auc2 else 2
    best_auc       = max(auc1, auc2)

    print(f"[Model 1 - Conservative] AUC: {auc1:.4f}  Accuracy: {acc1:.4f}")
    print(f"[Model 2 - Aggressive  ] AUC: {auc2:.4f}  Accuracy: {acc2:.4f}")
    print(f"=> Best: Model {best_model_num}  (AUC: {best_auc:.4f})")

    # evaluation.json 작성
    # PropertyFile이 이 파일을 읽어 BankAUCCondition 단계에서 best_auc 값을 추출합니다
    # json_path: "evaluation_metrics.best_auc.value"
    report = {
        "evaluation_metrics": {
            "model1_auc":      {"value": auc1,                  "standard_deviation": "NaN"},
            "model2_auc":      {"value": auc2,                  "standard_deviation": "NaN"},
            "model1_accuracy": {"value": acc1,                  "standard_deviation": "NaN"},
            "model2_accuracy": {"value": acc2,                  "standard_deviation": "NaN"},
            "best_auc":        {"value": best_auc,              "standard_deviation": "NaN"},
            "best_model":      {"value": float(best_model_num), "standard_deviation": "NaN"}
        }
    }
    os.makedirs(f"{BASE}/evaluation", exist_ok=True)
    with open(f"{BASE}/evaluation/evaluation.json", "w") as f:
        json.dump(report, f, indent=2)
    print("evaluation.json 저장 완료")

    # MLflow 로깅 (선택적)
    # 환경 변수로 ARN이 전달되지 않으면 조용히 건너뜁니다.
    # MLflow 실패가 파이프라인 전체를 중단시키지 않도록 try/except로 보호합니다.
    try:
        mlflow_tracking_arn = os.environ.get("MLFLOW_TRACKING_ARN")
        exp_name = os.environ.get("MLFLOW_EXPERIMENT_NAME")
        if mlflow_tracking_arn:
            import mlflow
            mlflow.set_tracking_uri(mlflow_tracking_arn)
            mlflow.set_experiment(exp_name or "bank-pipeline")
            with mlflow.start_run(run_name="pipeline-evaluation"):
                # 파이프라인 평가 결과를 MLflow에 기록
                mlflow.log_metrics({
                    "pipeline_model1_auc":      auc1,
                    "pipeline_model2_auc":      auc2,
                    "pipeline_model1_accuracy": acc1,
                    "pipeline_model2_accuracy": acc2,
                    "pipeline_best_auc":        best_auc,
                    "pipeline_best_model":      float(best_model_num)
                })
                # 어느 모델이 최선인지 태그로 기록
                mlflow.set_tag("best_model", f"model{best_model_num}")
            print("MLflow 로깅 완료")
    except Exception as e:
        print(f"MLflow 로깅 스킵 (선택적): {e}")


if __name__ == "__main__":
    main()

Overwriting pipeline_code/evaluation.py


## 4. 파이프라인 단계 정의

### Step 1: 데이터 전처리

**FrameworkProcessor(SKLearn)** 를 사용해 전처리 잡을 정의합니다. `pipeline_session`을 전달하면 `.run()` 호출 시 실제 잡이 시작되지 않고 **단계 정의만 생성**됩니다.

- **입력**: S3의 원본 CSV (`InputData` 파라미터)
- **출력**: train / validation / test / baseline 4개 폴더를 S3에 저장
- `step_preprocess.properties.ProcessingOutputConfig.Outputs["train"].S3Output.S3Uri` — 이후 훈련 단계에서 이 표현식으로 출력 경로를 **지연 참조**합니다. 실행 시점에 실제 S3 URI로 치환됩니다.

In [10]:
# ============================================================
# Step 1: 전처리 단계 (BankPreprocess) 정의
# ============================================================
# FrameworkProcessor(SKLearn): requirements.txt의 의존성을 자동 설치합니다.
# pipeline_session을 사용하므로 .run() 호출 시 실제 잡이 실행되지 않습니다.
sklearn_processor = FrameworkProcessor(
    estimator_cls=SKLearn,
    framework_version="1.2-1",    # scikit-learn 버전
    role=role,
    instance_type=processing_instance_type,  # 파이프라인 파라미터 참조
    instance_count=1,
    base_job_name="bank-pipeline-preprocess",
    sagemaker_session=pipeline_session        # 파이프라인 컨텍스트
)

processor_args = sklearn_processor.run(
    inputs=[
        ProcessingInput(
            source=input_data,            # InputData 파라미터 (S3 URI)
            destination="/opt/ml/processing/input",
            s3_data_distribution_type="FullyReplicated"  # 모든 인스턴스에 전체 데이터 복제
        )
    ],
    outputs=[
        # 각 output_name은 이후 단계에서 S3 URI를 참조할 때 키로 사용됩니다
        ProcessingOutput(output_name="train",      source="/opt/ml/processing/output/train"),
        ProcessingOutput(output_name="validation", source="/opt/ml/processing/output/validation"),
        ProcessingOutput(output_name="test",       source="/opt/ml/processing/output/test"),
        ProcessingOutput(output_name="baseline",   source="/opt/ml/processing/output/baseline"),
    ],
    code="pipeline_code/preprocessing.py",
    dependencies=["pipeline_code/requirements.txt"]  # pandas, numpy, scikit-learn 자동 설치
)

# ProcessingStep: 파이프라인 DAG에 등록할 단계 객체 생성
step_preprocess = ProcessingStep(name="BankPreprocess", step_args=processor_args)
print("BankPreprocess 단계 정의 완료")

sagemaker.config INFO - Applied value from config key = SageMaker.ProcessingJob.NetworkConfig.VpcConfig.Subnets
sagemaker.config INFO - Applied value from config key = SageMaker.ProcessingJob.NetworkConfig.VpcConfig.SecurityGroupIds
sagemaker.config INFO - Applied value from config key = SageMaker.TrainingJob.VpcConfig.Subnets
sagemaker.config INFO - Applied value from config key = SageMaker.TrainingJob.VpcConfig.SecurityGroupIds
sagemaker.config INFO - Applied value from config key = SageMaker.TrainingJob.Environment
BankPreprocess 단계 정의 완료


### Step 2: 모델 훈련 (Conservative / Aggressive 병렬)

두 XGBoost 모델을 **병렬**로 훈련합니다. 두 단계는 서로 의존성이 없으므로 SageMaker Pipelines가 동시에 실행합니다.

| 하이퍼파라미터 | Conservative (Model 1) | Aggressive (Model 2) |
|---|---|---|
| `eta` (학습률) | 0.5 (빠른 수렴) | 0.1 (세밀한 최적화) |
| `gamma` (최소 분할 손실) | 4 (보수적 분할) | 2 (공격적 분할) |
| `min_child_weight` | 6 (과적합 방지) | 3 (복잡한 패턴 허용) |
| `subsample` | 0.8 | 0.4 (강한 랜덤성) |
| `num_round` | 50 | 100 |

> 두 훈련 단계가 모두 완료된 후 BankEvaluate 단계가 실행됩니다. `step_train1.properties.ModelArtifacts.S3ModelArtifacts`로 모델 아티팩트 경로를 지연 참조합니다.

In [11]:
# ============================================================
# Step 2: 모델 훈련 단계 (Conservative / Aggressive 병렬) 정의
# ============================================================
# XGBoost 1.7-1 내장 알고리즘 컨테이너 URI 조회
xgb_image_uri = image_uris.retrieve(
    framework="xgboost", region=region,
    version="1.7-1", image_scope="training"
)

# 전처리 단계 출력의 S3 URI를 지연 참조
# 이 시점에는 문자열이 아닌 SageMaker 파이프라인 표현식 객체입니다.
# 실행 시점에 실제 S3 URI로 치환됩니다.
train_s3_uri = step_preprocess.properties.ProcessingOutputConfig.Outputs["train"].S3Output.S3Uri
val_s3_uri   = step_preprocess.properties.ProcessingOutputConfig.Outputs["validation"].S3Output.S3Uri

# ─────────────────────────────────────────────
# Model 1: Conservative (보수적 설정)
# eta=0.5로 빠르게 수렴, num_round=50으로 과적합 방지
# ─────────────────────────────────────────────
xgb_conservative = Estimator(
    image_uri=xgb_image_uri,
    instance_type=training_instance_type,  # 파이프라인 파라미터
    instance_count=1,
    output_path=f"s3://{bucket}/{prefix}/pipeline/output/model1/",
    role=role,
    base_job_name="bank-pipeline-conservative",
    sagemaker_session=pipeline_session
)
xgb_conservative.set_hyperparameters(
    max_depth=3,         # 트리 최대 깊이 (낮을수록 단순한 모델)
    eta=0.5,             # 학습률 (높을수록 빠른 수렴)
    gamma=4,             # 분할 최소 손실 감소량 (높을수록 보수적 분할)
    min_child_weight=6,  # 리프 노드 최소 가중치 합 (높을수록 과적합 방지)
    subsample=0.8,       # 각 트리 학습에 사용할 데이터 비율
    objective="binary:logistic",  # 이진 분류 (Bank Marketing 레이블: yes/no)
    eval_metric="auc",            # 검증 메트릭
    num_round=50,                 # 부스팅 반복 횟수
    verbosity=0                   # 훈련 로그 억제
)
train_args_1 = xgb_conservative.fit(
    inputs={
        "train":      TrainingInput(s3_data=train_s3_uri, content_type="text/csv"),
        "validation": TrainingInput(s3_data=val_s3_uri,   content_type="text/csv")
    }
)
# TrainingStep: 파이프라인 DAG에 등록
step_train1 = TrainingStep(name="BankTrainConservative", step_args=train_args_1)

# ─────────────────────────────────────────────
# Model 2: Aggressive (공격적 설정)
# eta=0.1로 천천히 학습, num_round=100으로 더 많은 반복
# ─────────────────────────────────────────────
xgb_aggressive = Estimator(
    image_uri=xgb_image_uri,
    instance_type=training_instance_type,
    instance_count=1,
    output_path=f"s3://{bucket}/{prefix}/pipeline/output/model2/",
    role=role,
    base_job_name="bank-pipeline-aggressive",
    sagemaker_session=pipeline_session
)
xgb_aggressive.set_hyperparameters(
    max_depth=3,
    eta=0.1,             # 낮은 학습률 → 세밀하게 최적화
    gamma=2,             # 더 공격적인 분할 허용
    min_child_weight=3,  # 더 작은 리프 허용 → 복잡한 패턴 학습
    subsample=0.4,       # 강한 랜덤 샘플링 → 다양성 증가
    objective="binary:logistic",
    eval_metric="auc",
    num_round=100,       # 더 많은 반복으로 세밀한 학습
    verbosity=0
)
train_args_2 = xgb_aggressive.fit(
    inputs={
        "train":      TrainingInput(s3_data=train_s3_uri, content_type="text/csv"),
        "validation": TrainingInput(s3_data=val_s3_uri,   content_type="text/csv")
    }
)
step_train2 = TrainingStep(name="BankTrainAggressive", step_args=train_args_2)

print("BankTrainConservative / BankTrainAggressive 단계 정의 완료")

sagemaker.config INFO - Applied value from config key = SageMaker.TrainingJob.VpcConfig.Subnets
sagemaker.config INFO - Applied value from config key = SageMaker.TrainingJob.VpcConfig.SecurityGroupIds
sagemaker.config INFO - Applied value from config key = SageMaker.TrainingJob.Environment
sagemaker.config INFO - Applied value from config key = SageMaker.TrainingJob.VpcConfig.Subnets
sagemaker.config INFO - Applied value from config key = SageMaker.TrainingJob.VpcConfig.SecurityGroupIds
sagemaker.config INFO - Applied value from config key = SageMaker.TrainingJob.Environment
BankTrainConservative / BankTrainAggressive 단계 정의 완료


### Step 3: 모델 평가 (두 모델 AUC 비교 + MLflow 로깅)

`ScriptProcessor`로 평가 스크립트를 실행합니다. 두 모델의 아티팩트와 테스트 데이터를 입력으로 받아 AUC/Accuracy를 계산하고 결과를 **`evaluation.json`** 으로 저장합니다.

`PropertyFile`은 파이프라인 런타임에 JSON 파일 내 특정 경로의 값을 읽어 조건 단계에서 사용할 수 있게 합니다.

```python
# 조건 단계에서 이렇게 참조됩니다
JsonGet(step_name="BankEvaluate", property_file=evaluation_report,
        json_path="evaluation_metrics.best_auc.value")
```

MLflow 환경 변수(`MLFLOW_TRACKING_ARN`, `MLFLOW_EXPERIMENT_NAME`)는 파이프라인 파라미터 값이 런타임에 치환되어 컨테이너에 전달됩니다.

In [12]:
# ============================================================
# Step 3: 모델 평가 단계 (BankEvaluate) 정의
# ============================================================
# ScriptProcessor: 커스텀 스크립트를 지정 컨테이너에서 실행합니다.
# XGBoost 컨테이너를 재사용하여 별도 컨테이너 빌드 없이 xgboost 라이브러리를 활용합니다.
eval_processor = ScriptProcessor(
    image_uri=xgb_image_uri,
    command=["python3"],
    instance_type=processing_instance_type,
    instance_count=1,
    base_job_name="bank-pipeline-evaluate",
    role=role,
    # MLflow 추적 서버 정보를 환경 변수로 컨테이너에 전달
    # 파이프라인 파라미터 값이 런타임에 치환됩니다
    env={
        "MLFLOW_TRACKING_ARN": mlflow_tracking_arn,
        "MLFLOW_EXPERIMENT_NAME": mlflow_experiment
    },
    sagemaker_session=pipeline_session
)

# 테스트 데이터 S3 URI (전처리 단계 출력 참조)
test_s3_uri = step_preprocess.properties.ProcessingOutputConfig.Outputs["test"].S3Output.S3Uri

eval_args = eval_processor.run(
    inputs=[
        # 두 훈련 단계의 모델 아티팩트를 각각 다른 경로에 마운트
        ProcessingInput(
            source=step_train1.properties.ModelArtifacts.S3ModelArtifacts,  # model.tar.gz
            destination="/opt/ml/processing/model1"
        ),
        ProcessingInput(
            source=step_train2.properties.ModelArtifacts.S3ModelArtifacts,
            destination="/opt/ml/processing/model2"
        ),
        ProcessingInput(
            source=test_s3_uri,
            destination="/opt/ml/processing/test"
        )
    ],
    outputs=[
        ProcessingOutput(
            output_name="evaluation",
            source="/opt/ml/processing/evaluation"   # evaluation.json이 저장되는 경로
        )
    ],
    code="pipeline_code/evaluation.py"
)

# PropertyFile: 파이프라인이 런타임에 S3에서 JSON 파일을 읽어 조건 평가에 활용합니다.
# BankAUCCondition 단계에서 JsonGet()으로 best_auc 값을 추출합니다.
evaluation_report = PropertyFile(
    name="EvaluationReport",
    output_name="evaluation",          # ProcessingOutput의 output_name과 일치해야 합니다
    path="evaluation.json"             # 출력 S3 URI 내 파일 경로
)

step_eval = ProcessingStep(
    name="BankEvaluate",
    step_args=eval_args,
    property_files=[evaluation_report]  # 이 파일을 파이프라인 조건 단계에서 참조 가능하게 등록
)

print("BankEvaluate 단계 정의 완료")

sagemaker.config INFO - Applied value from config key = SageMaker.ProcessingJob.NetworkConfig.VpcConfig.Subnets
sagemaker.config INFO - Applied value from config key = SageMaker.ProcessingJob.NetworkConfig.VpcConfig.SecurityGroupIds
BankEvaluate 단계 정의 완료


### Step 4: 조건부 모델 등록 (AUC ≥ 임계값)

평가 단계의 `best_auc` 값과 `AucThreshold` 파라미터를 비교하여 분기합니다.

```
best_auc >= AucThreshold (0.75)
    ├─ True  → BankRegisterModel: BankMarketingModelPackageGroup에 모델 패키지 등록
    └─ False → BankPipelineFail: 파이프라인을 명시적 실패로 종료
```

- 등록 모델에는 `evaluation.json`의 메트릭이 `ModelMetrics`로 첨부됩니다.
- 초기 승인 상태는 `ModelApprovalStatus` 파라미터로 제어합니다 (기본: `PendingManualApproval`).
- `FailStep`의 오류 메시지는 `Join()`으로 동적으로 구성되어 실패 원인을 명확히 기록합니다.

In [13]:
from sagemaker.model_metrics import ModelMetrics, MetricsSource

# Model Registry 패키지 그룹 이름 (없으면 자동 생성됨)
model_package_group = "BankMarketingModelPackageGroup"

# 평가 결과 JSON의 S3 URI를 모델 메트릭으로 첨부
eval_output_uri = step_eval.arguments["ProcessingOutputConfig"]["Outputs"][0]["S3Output"]["S3Uri"]
model_metrics = ModelMetrics(
    model_statistics=MetricsSource(
        s3_uri=f"{eval_output_uri}/evaluation.json",
        content_type="application/json"
    )
)

def make_register_step(name, step_train):
    """훈련 단계의 아티팩트를 Model Registry에 등록하는 ModelStep을 생성합니다."""
    m = Model(
        image_uri=xgb_image_uri,
        model_data=step_train.properties.ModelArtifacts.S3ModelArtifacts,
        role=role,
        sagemaker_session=pipeline_session
    )
    return ModelStep(name=name, step_args=m.register(
        content_types=["text/csv"],
        response_types=["text/csv"],
        inference_instances=["ml.m5.large", "ml.m5.xlarge"],
        transform_instances=["ml.m5.large"],
        model_package_group_name=model_package_group,
        approval_status=model_approval_status,
        model_metrics=model_metrics
    ))

# evaluation.py가 best_model.value = 1.0(Conservative) 또는 2.0(Aggressive)으로 기록
# BankChooseModel: best_model == 1.0이면 Model1 등록, 아니면 Model2 등록
step_register1 = make_register_step("BankRegisterModel",  step_train1)  # Conservative
step_register2 = make_register_step("BankRegisterModel2", step_train2)  # Aggressive

cond_model1_best = ConditionEquals(
    left=JsonGet(
        step_name=step_eval.name,
        property_file=evaluation_report,
        json_path="evaluation_metrics.best_model.value"  # 1.0 또는 2.0
    ),
    right=1.0  # 1.0이면 Conservative가 더 나은 모델
)

step_choose_model = ConditionStep(
    name="BankChooseModel",
    conditions=[cond_model1_best],
    if_steps=[step_register1],   # Model 1이 best → Conservative 등록
    else_steps=[step_register2]  # Model 2가 best → Aggressive 등록
)

# ─────────────────────────────────────────────
# 실패 단계: AUC 임계값 미달 시 파이프라인을 명시적으로 실패시킵니다
# ─────────────────────────────────────────────
step_fail = FailStep(
    name="BankPipelineFail",
    error_message=Join(
        on=" ",
        values=["AUC 임계값 미달:", auc_threshold, "기준 통과 필요"]
    )
)

# 외부 조건: best_auc >= AucThreshold 통과 시 → BankChooseModel (최적 모델 선택)
# 미달 시 → BankPipelineFail
cond_auc = ConditionGreaterThanOrEqualTo(
    left=JsonGet(
        step_name=step_eval.name,
        property_file=evaluation_report,
        json_path="evaluation_metrics.best_auc.value"
    ),
    right=auc_threshold
)

step_condition = ConditionStep(
    name="BankAUCCondition",
    conditions=[cond_auc],
    if_steps=[step_choose_model],  # AUC 통과 → 최적 모델 선택 후 등록
    else_steps=[step_fail]         # AUC 미달 → 파이프라인 실패
)

print("BankAUCCondition / BankChooseModel / BankRegisterModel(1/2) / BankPipelineFail 정의 완료")


sagemaker.config INFO - Applied value from config key = SageMaker.Model.VpcConfig
sagemaker.config INFO - Applied value from config key = SageMaker.Model.VpcConfig
BankAUCCondition / BankChooseModel / BankRegisterModel(1/2) / BankPipelineFail 정의 완료


## 5. 파이프라인 생성 및 등록

정의된 모든 단계와 파라미터를 묶어 파이프라인 객체를 생성하고 SageMaker에 등록(upsert)합니다.

`pipeline.upsert()`는 같은 이름의 파이프라인이 없으면 생성, 있으면 업데이트합니다. 이 시점에 파이프라인 DAG 정의가 AWS에 저장되지만 실행은 시작되지 않습니다.

> `steps` 목록에 나열된 단계 중 의존성이 자동 추론됩니다 — `step_train1`과 `step_train2`는 `step_preprocess`의 출력을 참조하므로 자동으로 직렬 연결됩니다. 서로 간 의존성이 없는 두 훈련 단계는 병렬 실행됩니다.

In [14]:
# ============================================================
# 파이프라인 객체 생성 및 AWS에 등록
# ============================================================
pipeline_name = "BankMarketingPipeline"

pipeline = Pipeline(
    name=pipeline_name,
    # 파이프라인 파라미터: 실행 시 외부에서 값을 전달할 수 있습니다
    parameters=[
        processing_instance_type,
        training_instance_type,
        model_approval_status,
        auc_threshold,
        input_data,
        mlflow_tracking_arn,
        mlflow_experiment
    ],
    # 단계 목록: SageMaker Pipelines가 의존성을 분석하여 실행 순서를 자동 결정합니다
    # BankPreprocess → (BankTrainConservative || BankTrainAggressive) → BankEvaluate
    # → BankAUCCondition → BankChooseModel → BankRegisterModel(1 or 2)
    steps=[
        step_preprocess,
        step_train1,
        step_train2,
        step_eval,
        step_condition
    ],
    sagemaker_session=pipeline_session
)

# upsert: 동일 이름의 파이프라인이 존재하면 업데이트, 없으면 신규 생성
# 이 호출로 파이프라인 정의가 AWS에 저장되지만 실행은 시작되지 않습니다
pipeline.upsert(role_arn=role)
print(f"파이프라인 등록 완료: {pipeline_name}")

sagemaker.config INFO - Applied value from config key = SageMaker.TrainingJob.VpcConfig.Subnets
sagemaker.config INFO - Applied value from config key = SageMaker.TrainingJob.VpcConfig.SecurityGroupIds
sagemaker.config INFO - Applied value from config key = SageMaker.TrainingJob.Environment
sagemaker.config INFO - Applied value from config key = SageMaker.TrainingJob.VpcConfig.Subnets
sagemaker.config INFO - Applied value from config key = SageMaker.TrainingJob.VpcConfig.SecurityGroupIds
sagemaker.config INFO - Applied value from config key = SageMaker.TrainingJob.Environment
파이프라인 등록 완료: BankMarketingPipeline


## 6. 파이프라인 실행

`pipeline.start()`로 파이프라인 실행을 트리거합니다. 반환된 `execution` 객체로 실행 상태를 추적합니다.

- `execution.arn`: 이번 실행의 고유 식별자 (CloudWatch, SageMaker Studio에서 추적 가능)
- `execution.wait()`: 완료/실패까지 블로킹 대기 — 실패 시 `WaiterError` 발생
- `execution.list_steps()`: 각 단계의 상태, 시작/종료 시간, 실패 이유 조회
- `execution.describe()`: 전체 실행 상태 및 실패 이유 조회

> 파이프라인 실행 중에도 노트북 커널은 계속 사용 가능합니다. 완료를 기다리지 않고 위 셀에서 멈춰도 파이프라인은 AWS 클라우드에서 계속 실행됩니다.

In [15]:
# ============================================================
# 파이프라인 실행 시작
# ============================================================
# pipeline.start()는 즉시 반환되며, 파이프라인은 백그라운드에서 실행됩니다.
# 파라미터를 전달하지 않으면 파이프라인 정의의 기본값이 사용됩니다.
execution = pipeline.start(
    parameters={
        "AucThreshold": 0.75,                         # 모델 등록 최소 AUC 기준
        "ModelApprovalStatus": "PendingManualApproval" # 등록 후 수동 검토 필요
    }
)

print(f"파이프라인 실행 ARN: {execution.arn}")
print("실행 완료까지 약 15~20분 소요됩니다...")

파이프라인 실행 ARN: arn:aws:sagemaker:us-west-2:975050309668:pipeline/BankMarketingPipeline/execution/ean6qbytvggt
실행 완료까지 약 15~20분 소요됩니다...


In [16]:
# ============================================================
# 파이프라인 실행 상태 확인 (현재 스냅샷)
# ============================================================
execution_details = execution.describe()
print(f"실행 상태: {execution_details['PipelineExecutionStatus']}")
print(f"실패 이유: {execution_details.get('FailureReason', 'N/A')}")

# 각 단계별 상세 정보 확인
# StepStatus: Executing, Succeeded, Failed, Stopped, Cached 중 하나
steps = execution.list_steps()
for step in steps:
    print(f"\n단계: {step['StepName']}")
    print(f"상태: {step['StepStatus']}")
    if step['StepStatus'] == 'Failed':
        print(f"실패 이유: {step.get('FailureReason', 'N/A')}")
        # Metadata에는 실제 잡 ARN이 포함되어 CloudWatch 로그를 직접 조회할 수 있습니다
        if 'Metadata' in step:
            print(f"메타데이터: {step['Metadata']}")

실행 상태: Executing
실패 이유: N/A


In [17]:
# ============================================================
# 파이프라인 실행 완료 대기 (블로킹)
# ============================================================
# execution.wait()는 파이프라인이 Succeeded 또는 Failed 상태가 될 때까지 대기합니다.
# 파이프라인이 Failed 상태로 종료되면 WaiterError 예외가 발생합니다.
# try/except로 감싸 정상 종료와 실패 종료를 모두 처리합니다.
from botocore.exceptions import WaiterError

try:
    execution.wait()
    print("파이프라인 실행 완료: Succeeded")
except WaiterError:
    # 파이프라인이 Failed 상태이면 WaiterError가 발생합니다
    # 실제 실패 원인은 describe() 또는 list_steps()로 확인합니다
    desc = execution.describe()
    print(f"파이프라인 실행 종료: {desc['PipelineExecutionStatus']}")
    print(f"실패 이유: {desc.get('FailureReason', 'N/A')}")
    print("\n단계별 상태 확인:")
    for step in execution.list_steps():
        if step["StepStatus"] == "Failed":
            print(f"  [FAILED] {step['StepName']}: {step.get('FailureReason', 'N/A')}")

파이프라인 실행 완료: Succeeded


## 7. 실행 결과 확인

파이프라인 실행 결과를 세 가지 방식으로 확인합니다.

1. **단계별 상태표** — 각 단계의 성공/실패 여부와 실행 시간
2. **evaluation.json** — BankEvaluate 단계가 S3에 저장한 두 모델의 AUC/Accuracy 비교
3. **MLflow 실험** — `pipeline-evaluation` run에 기록된 지표

> BankEvaluate 단계가 성공한 경우에만 evaluation.json을 조회할 수 있습니다. BankPreprocess나 BankTrain 단계가 실패하면 평가 단계가 실행되지 않으므로 `"평가 단계가 아직 완료되지 않았거나 실패했습니다."` 메시지가 출력됩니다.

In [18]:
# ============================================================
# 각 단계별 실행 결과 요약
# ============================================================
import pandas as pd

steps = execution.list_steps()
step_df = pd.DataFrame([
    {
        "단계": s["StepName"],
        "상태": s["StepStatus"],
        "시작": s.get("StartTime", ""),   # 단계 실행 시작 시각
        "종료": s.get("EndTime", "")      # 단계 완료 시각 (실행 중이면 빈 값)
    }
    for s in steps
])
print(step_df.to_string(index=False))

                              단계        상태                               시작                               종료
BankRegisterModel2-RegisterModel Succeeded 2026-06-21 17:35:13.888000+00:00 2026-06-21 17:35:14.960000+00:00
                 BankChooseModel Succeeded 2026-06-21 17:35:13.373000+00:00 2026-06-21 17:35:13.515000+00:00
                BankAUCCondition Succeeded 2026-06-21 17:35:12.407000+00:00 2026-06-21 17:35:12.835000+00:00
                    BankEvaluate Succeeded 2026-06-21 17:30:03.367000+00:00 2026-06-21 17:35:11.824000+00:00
             BankTrainAggressive Succeeded 2026-06-21 17:26:54.565000+00:00 2026-06-21 17:30:02.972000+00:00
           BankTrainConservative Succeeded 2026-06-21 17:26:54.565000+00:00 2026-06-21 17:29:59.630000+00:00
                  BankPreprocess Succeeded 2026-06-21 17:21:50.421000+00:00 2026-06-21 17:26:53.885000+00:00


In [23]:
# ============================================================
# evaluation.json 다운로드 및 모델 메트릭 출력
# ============================================================
import json
from sagemaker.s3 import S3Downloader

# BankEvaluate 단계를 단계 목록에서 찾기
eval_step = next(
    (s for s in steps if s["StepName"] == "BankEvaluate"), None
)

if eval_step and eval_step["StepStatus"] == "Succeeded":
    # 단계 메타데이터에서 Processing 잡 ARN을 추출합니다
    output_s3 = eval_step["Metadata"]["ProcessingJob"]["Arn"]
    job_name  = output_s3.split("/")[-1]
    # 잡 상세 정보에서 출력 S3 URI를 조회합니다
    desc      = sm_client.describe_processing_job(ProcessingJobName=job_name)
    eval_uri  = next(
        o["S3Output"]["S3Uri"]
        for o in desc["ProcessingOutputConfig"]["Outputs"]
        if o["OutputName"] == "evaluation"
    )
    # S3에서 evaluation.json을 직접 읽어 메트릭 출력
    report = json.loads(S3Downloader.read_file(f"{eval_uri}/evaluation.json"))
    metrics = report["evaluation_metrics"]

    print("=" * 45)
    print(f"  Model 1 (Conservative) AUC      : {metrics['model1_auc']['value']:.4f}")
    print(f"  Model 1 (Conservative) Accuracy : {metrics['model1_accuracy']['value']:.4f}")
    print(f"  Model 2 (Aggressive)   AUC      : {metrics['model2_auc']['value']:.4f}")
    print(f"  Model 2 (Aggressive)   Accuracy : {metrics['model2_accuracy']['value']:.4f}")
    print("-" * 45)
    print(f"  Best AUC  : {metrics['best_auc']['value']:.4f}")
    print(f"  Best Model: Model {int(metrics['best_model']['value'])}")
    print("=" * 45)
else:
    print("평가 단계가 아직 완료되지 않았거나 실패했습니다.")

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
  Model 1 (Conservative) AUC      : 0.7907
  Model 1 (Conservative) Accuracy : 0.8990
  Model 2 (Aggressive)   AUC      : 0.7932
  Model 2 (Aggressive)   Accuracy : 0.9029
---------------------------------------------
  Best AUC  : 0.7932
  Best Model: Model 2


In [20]:
# ============================================================
# MLflow 실험 결과 확인: pipeline-evaluation 런 조회
# ============================================================
mlflow.set_tracking_uri(mlflow_arn)
client = mlflow.tracking.MlflowClient()
exp    = client.get_experiment_by_name(experiment_name)

if exp:
    runs = client.search_runs(
        experiment_ids=[exp.experiment_id],
        filter_string="tags.mlflow.runName = 'pipeline-evaluation'",
        order_by=["start_time DESC"],
        max_results=1
    )
    if runs:
        r = runs[0]
        best_auc_val   = r.data.metrics.get("pipeline_best_auc")
        best_model_val = r.data.metrics.get("pipeline_best_model")
        print(f"Run ID    : {r.info.run_id}")
        # 메트릭이 없을 경우 'N/A'를 float 포맷으로 출력하면 ValueError 발생
        # get()이 None을 반환할 때 조건부 포맷을 적용합니다
        print(f"Best AUC  : {best_auc_val:.4f}" if best_auc_val is not None else "Best AUC  : N/A")
        print(f"Best Model: Model {int(best_model_val)}" if best_model_val is not None else "Best Model: N/A")
    else:
        print("pipeline-evaluation run을 찾을 수 없습니다.")
else:
    print(f"실험 '{experiment_name}'을 찾을 수 없습니다.")


pipeline-evaluation run을 찾을 수 없습니다.


---
## ✅ 최종 실행 결과 (2026-06-21)

### 실행 정보

| 항목 | 값 |
|------|----|
| 실행 ARN | `...pipeline/BankMarketingPipeline/execution/3h6lta6aesj4` |
| 총 소요 시간 | 약 11분 |
| 파이프라인 상태 | **Succeeded** |

### 단계별 실행 현황

| 단계 | 상태 | 소요 |
|------|------|------|
| BankPreprocess | ✅ Succeeded | 3분 |
| BankTrainConservative / Aggressive | ✅ Succeeded | 3분 (병렬) |
| BankEvaluate | ✅ Succeeded | 5분 |
| BankAUCCondition → BankChooseModel → **BankRegisterModel2** | ✅ Succeeded | 즉시 |

### 모델 평가 결과

| 모델 | AUC | Accuracy | 선택 |
|------|-----|----------|------|
| Model 1 — Conservative (eta=0.5, round=50) | 0.7907 | 0.8990 | |
| Model 2 — Aggressive (eta=0.1, round=100) | **0.7932** | **0.9029** | ✅ Best |

**Best: Model 2 (Aggressive), AUC = 0.7932** — 임계값 0.75 통과

### 조건 단계 분기

```
BankAUCCondition (0.7932 >= 0.75) → True
  └─ BankChooseModel (best_model == 1.0 ?) → False  ← Model 2가 best
       └─ BankRegisterModel2 ✅  (Aggressive 아티팩트 등록)
```

### MLflow pipeline-evaluation Run

| 메트릭 | 값 |
|--------|----|
| `pipeline_model1_auc` | 0.7907 |
| `pipeline_model2_auc` | **0.7932** |
| `pipeline_model1_accuracy` | 0.8990 |
| `pipeline_model2_accuracy` | **0.9029** |
| `pipeline_best_auc` | **0.7932** |
| `pipeline_best_model` | **2** (Aggressive) |

> Run ID: `41e8879b740d4bb8...`

### 코드 리뷰 수정 이력

| # | 수정 내용 | 효과 |
|---|-----------|------|
| 1 | `y_no` 레이블 누수 제거 | AUC 1.0000 → **0.7932** (실제값) |
| 2 | `BankChooseModel` 추가 | best_model에 따라 올바른 아티팩트 등록 |
| 3 | MLflow `--target` 설치 | XGBoost 컨테이너에서 MLflow 로깅 성공 |
| 4 | `WaiterError` 처리 | 파이프라인 실패 시 크래시 없이 원인 출력 |
| 5 | 전처리 `sep=","` 수정 | CSV 구분자 오류로 인한 `KeyError: pdays` 해결 |


## 8. 리소스 정리 (선택)

파이프라인 실행이 완료된 후 불필요한 리소스를 정리합니다.

In [21]:
# ============================================================
# 리소스 정리 (선택적)
# ============================================================
# DELETE_PIPELINE을 True로 변경하면 BankMarketingPipeline이 삭제됩니다.
# 파이프라인 삭제 후에도 실행 기록과 모델 아티팩트는 S3/Model Registry에 남습니다.
# 완전 정리가 필요하면 S3 경로와 모델 패키지 그룹도 별도로 삭제하세요.
DELETE_PIPELINE = False  # True로 변경 시 파이프라인 삭제

if DELETE_PIPELINE:
    sm_client.delete_pipeline(PipelineName=pipeline_name)
    print(f"파이프라인 삭제 완료: {pipeline_name}")
else:
    print("DELETE_PIPELINE = False — 파이프라인 유지")

DELETE_PIPELINE = False — 파이프라인 유지
